In [20]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
if os.path.basename(PROJECT_ROOT) == "notebooks":
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [21]:
import torch
import torch.nn as nn
from torchmetrics.text import WordErrorRate


from srcs.nets.e2e import get_model as create_model
from srcs.nets.lora import apply_lora
from srcs.nets.utils import ctc_decode, freeze, load_weights
from srcs.nets.backend.refiner.refiner import MyRefiner
from srcs.nets.backend.nets_utils import make_non_pad_mask

from srcs.spm.spm_train import ensure_unigram
from srcs.spm.text_transofm import TextTransform

from srcs.trainer.trainer import HFTrainer
from transformers import EarlyStoppingCallback, TrainingArguments
from srcs.datasets.vicocktail import Collator, load_vicocktail

In [22]:
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "refiner_p_only")

BASE_MODEL_CHECKPOINT = os.path.join(PROJECT_ROOT, "checkpoints", "vsr_lora_6", "final")
TEACHER_MODEL_CHECKPOINT = os.path.join(PROJECT_ROOT, "checkpoints", "vsr_lora_12", "checkpoint-125500")

os.makedirs(OUTPUT_PATH, exist_ok=True)

EPOCHS = 20
SEED = 42
NUM_WORKERS = 0
MAX_GRAD_NORM = 5.0
WEIGHT_DECAY = 0.005
LR = 0.0001
GRADIENT_PER_STEPS = 2
BATCH = 30

In [23]:
dataset_splits = load_vicocktail(
    train_fraction=1.0,
    validation_fraction=1.0,
    test_fraction=1.0,
    splits=("train", "val", "test"),
    seed=SEED,
)

train_dataset = dataset_splits["train"]
validation_dataset = dataset_splits["val"]
test_dataset = dataset_splits["test"]

print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(validation_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

Train samples: 188,229
Validation samples: 5,844
Test samples: 1,167


In [24]:
model_path, units_path = ensure_unigram(train_dataset)
text_transform = TextTransform(model_path, units_path)

In [25]:
def load_vsr_lora(vocab_size, checkpoint_path, model_size="small"):
    model = create_model("auto-vsr", vocab_size, size=model_size)
    apply_lora(
        model=model,
        start_block=0,
        rank=16,
        alpha=16,
        dropout_rate=0.05,
        target_modules=("linear_q", "linear_v"),
    )
    report = load_weights(model, checkpoint_path)
    freeze(model)
    print(f"Loaded base model tensors: {report['loaded']:,}")
    return model

In [26]:
base_model = load_vsr_lora(text_transform.vocab_size, BASE_MODEL_CHECKPOINT)
#teacher_model = load_vsr_lora(text_transform.vocab_size, TEACHER_MODEL_CHECKPOINT, "large")

Loaded base model tensors: 390


In [ ]:
class CTCOnlyRefinerModel(nn.Module):
    def __init__(self, model, vocab_size, k=0):
        super().__init__()
        self.model = model
        self.refiner = MyRefiner(vocab_size=vocab_size, k=k)
        freeze(self.model)

    def train(self, mode=True):
        super().train(mode)
        self.model.eval()
        return self

    def forward(self, videos, video_lengths, labels=None, label_lengths=None):
        with torch.no_grad():
            contexts = self.model.get_contexts(videos, video_lengths)
            base_logits = contexts["logits"]

        mask = make_non_pad_mask(contexts["input_lengths"]).to(videos.device)
        outputs = self.refiner(logits=base_logits, visual_feats=None, mask=mask)

        new_logits = outputs["logits_steps"][-1]

        loss = self.model.ctc.loss_from_logits(
            new_logits, contexts["input_lengths"], labels, label_lengths
        )

        return {
            "loss": loss,
            "logits": new_logits,
            "input_lengths": contexts["input_lengths"],
            "base_logits": base_logits,
        }

In [ ]:
class CustomTrainer(HFTrainer):
    def __init__(self, *args, text_transform, **kwargs):
        super().__init__(*args, **kwargs)
        self.text_transform = text_transform
        self.output_metrics = {"train": self._metrics(), "eval": self._metrics()}

    @staticmethod
    def _metrics():
        return {
            "wer": WordErrorRate(),
            "base_wer": WordErrorRate(),
            "sample_count": 0,
            "flip_count": 0,
            "frame_count": 0,
        }

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        loss, outputs = super().compute_loss(
            model=model,
            inputs=inputs,
            return_outputs=True,
            num_items_in_batch=num_items_in_batch,
        )

        mode = "train" if model.training else "eval"
        with torch.no_grad():
            self._update_metrics(outputs, inputs, mode)

        if return_outputs:
            return loss, outputs

        return loss

    def _update_metrics(self, outputs, inputs, mode):
        new_logits = outputs["logits"]
        base_logits = outputs["base_logits"]
        input_lengths = outputs["input_lengths"]
        valid_mask = make_non_pad_mask(input_lengths).to(new_logits.device)

        state = self.output_metrics[mode]

        frame_flips = base_logits.argmax(dim=-1) != new_logits.argmax(dim=-1)

        state["flip_count"] += int(frame_flips[valid_mask].sum().item())
        state["frame_count"] += int(valid_mask.sum().item())

        references = self._decode_references(inputs)
        hypotheses = self._decode_logits(new_logits, input_lengths)
        state["wer"].update(hypotheses, references)
        state["sample_count"] += len(references)

        if mode == "eval":
            base_hypotheses = self._decode_logits(base_logits, input_lengths)
            state["base_wer"].update(base_hypotheses, references)

    def _decode_logits(self, logits, input_lengths):
        token_ids = ctc_decode(logits, input_lengths, self.text_transform.blank_id)
        return [self.text_transform.decode(ids) for ids in token_ids]

    def _decode_references(self, inputs):
        labels = inputs["labels"].detach().cpu()
        label_lengths = inputs["label_lengths"].detach().cpu().tolist()
        return [
            self.text_transform.decode(label[:length])
            for label, length in zip(labels, label_lengths)
        ]

    def log(self, logs, start_time=None):
        if "eval_loss" in logs:
            self._add_output_metrics(logs, "eval")
        elif "loss" in logs or "train_loss" in logs:
            self._add_output_metrics(logs, "train")

        super().log(logs, start_time=start_time)

    def _add_output_metrics(self, logs, mode):
        state = self.output_metrics[mode]
        if state["sample_count"] == 0:
            return

        prefix = "eval_" if mode == "eval" else ""
        logs[prefix + "wer"] = state["wer"].compute().item()
        logs[prefix + "refined_flip_rate"] = state["flip_count"] / state["frame_count"]

        if mode == "eval":
            logs["eval_base_wer"] = state["base_wer"].compute().item()

        self.output_metrics[mode] = self._metrics()

In [28]:
train_config = TrainingArguments(
    output_dir=OUTPUT_PATH,
    logging_dir=os.path.join(OUTPUT_PATH, "logs"),
    label_names=["labels", "label_lengths"],
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    gradient_accumulation_steps=GRADIENT_PER_STEPS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.05,
    max_grad_norm=MAX_GRAD_NORM,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    dataloader_num_workers=NUM_WORKERS,
    dataloader_pin_memory=torch.cuda.is_available(),
    dataloader_persistent_workers=NUM_WORKERS > 0,
    dataloader_prefetch_factor=2 if NUM_WORKERS > 0 else None,
    train_sampling_strategy="group_by_length",
    length_column_name="video_length",
    load_best_model_at_end=True,
    metric_for_best_model="eval_wer",
    greater_is_better=False,
    save_total_limit=3,
    seed=SEED,
    data_seed=SEED,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
model = CTCOnlyRefinerModel(
    model=base_model,
    vocab_size=text_transform.vocab_size,
    k=0,
)

trainer = CustomTrainer(
    model=model,
    args=train_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    train_collator=Collator(text_transform, "train"),
    validation_collator=Collator(text_transform, "val"),
    text_transform=text_transform,
    allowed_trainable_names=("refiner.",),
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.001,
        )
    ],
)

In [ ]:
train_result = trainer.train()
train_result

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset)
test_metrics = {
    key.replace("eval_", "test_", 1): value
    for key, value in test_metrics.items()
}
test_metrics